### Problem Understanding

#### What is this problem?
Type: Supervised Machine Learning

Task: Regression

Target Variable: SalePrice

Evaluation Metric: RMSE (Root Mean Squared Error) on log(SalePrice)

##### Why log?
House prices are right-skewed

Log transformation stabilizes variance and improves model performance

### Dataset Overview
| File                    | Purpose                        |
| ----------------------- | ------------------------------ |
| `train.csv`             | Training data (with SalePrice) |
| `test.csv`              | Test data (without SalePrice)  |
| `sample_submission.csv` | Submission format              |
| `data_description.txt`  | Column explanations            |

##### Size
Train: 1460 rows

Test: 1459 rows

Features: 79

In [ ]:
#### Imports & Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")


In [ ]:
## Load the Data
train_df = pd.read_csv("../files/house-prices-train.csv")
test_df = pd.read_csv("../files/house-prices-test.csv")

print(train_df.shape)
print(test_df.shape)

#### Initial Exploration (EDA)

In [ ]:
## Target Variable Distribution
sns.histplot(train_df['SalePrice'], kde=True)
plt.show()
train_df["SalePrice"].skew()

In [ ]:
train_df["SalePrice"] = np.log1p(train_df["SalePrice"])

In [ ]:
sns.histplot(train_df["SalePrice"], kde=True)
plt.show()


In [ ]:
## Missing Values
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing.head(10)

In [ ]:
## Combine Train & Test for Preprocessing
y = train_df["SalePrice"]
train_df.drop("SalePrice", axis=1, inplace=True)
full_df = pd.concat([train_df, test_df], axis=0)

In [ ]:
print(y.shape)
print(train_df.shape)
print(full_df.shape)

In [ ]:
## Handling Missing Values
## Categorical: Missing = “None”
cat_none = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]

for col in cat_none:
    full_df[col] = full_df[col].fillna("None")

In [ ]:
## Numerical Missing
full_df["LotFrontage"] = full_df.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

num_zero = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"
]

for col in num_zero:
    full_df[col] = full_df[col].fillna(0)


In [ ]:
## Mode for Remaining
for col in full_df.columns:
    if full_df[col].isnull().sum() > 0:
        full_df[col] = full_df[col].fillna(full_df[col].mode()[0])


### Feature Engineering

In [ ]:
## Create New Features
full_df["TotalSF"] = (
    full_df["TotalBsmtSF"]
    + full_df["1stFlrSF"]
    + full_df["2ndFlrSF"]
)

full_df["HouseAge"] = full_df["YrSold"] - full_df["YearBuilt"]
full_df["RemodAge"] = full_df["YrSold"] - full_df["YearRemodAdd"]


In [ ]:
## Ordinal Encoding (Quality Features)
quality_map = {
    "Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0
}

qual_cols = [
    "ExterQual", "ExterCond", "HeatingQC",
    "KitchenQual", "FireplaceQu",
    "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond"
]

for col in qual_cols:
    full_df[col] = full_df[col].map(quality_map)


In [ ]:
## One-Hot Encoding
full_df = pd.get_dummies(full_df, drop_first=True)

In [ ]:
## Split Back into Train & Test
X = full_df.iloc[:len(y), :]
X_test = full_df.iloc[len(y):, :]


In [ ]:
## Train–Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Modeling

In [ ]:
## Baseline: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

preds = lr.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse


In [ ]:
## Ridge Regression
ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train)

preds = ridge.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse


In [ ]:
## Lasso (Feature Selection)
lasso = Lasso(alpha=0.0005)
lasso.fit(X_train, y_train)

preds = lasso.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse


In [ ]:
## Gradient Boosting (Very Strong)
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

gbr.fit(X_train, y_train)

preds = gbr.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse


In [ ]:
## Train Final Model on Full Data
final_model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    random_state=42
)

final_model.fit(X, y)

In [ ]:
## Predict Test Data
test_preds = final_model.predict(X_test)
test_preds = np.expm1(test_preds)  # reverse log


In [ ]:
## Create Submission File
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_preds
})

submission.to_csv("../files/house-price-prediction.csv", index=False)


##### Steps to Reduce RMSE

In [ ]:
## Cross-Validation
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse = -cross_val_score(
    gbr, X, y,
    scoring="neg_root_mean_squared_error",
    cv=kf
)

rmse.mean()


In [ ]:
## Gradient Boosting tuning example
gbr = GradientBoostingRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=3,
    min_samples_leaf=15,
    min_samples_split=10,
    subsample=0.8,
    random_state=42
)

gbr.fit(X_train, y_train)

preds = gbr.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse

In [ ]:
## Cross-Validation after GBR tuning
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse = -cross_val_score(
    gbr, X, y,
    scoring="neg_root_mean_squared_error",
    cv=kf
)

rmse.mean()

In [ ]:
## XG Booster
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=3,
    subsample=0.7,
    colsample_bytree=0.7,
    objective="reg:squarederror",
    random_state=42
)

xgb.fit(X_train, y_train)

preds = xgb.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse

In [ ]:
## LightGBM
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb.fit(X_train, y_train)

preds = xgb.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse